In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType


airport_schema = StructType([
    StructField("id",         StringType(), True),
    StructField("name",       StringType(), True),
    StructField("city",       StringType(), True),
    StructField("country",    StringType(), True),
    StructField("iata",       StringType(), True),
    StructField("icao",       StringType(), True),
    StructField("latitude",   DoubleType(), True),
    StructField("longitude",  DoubleType(), True),
    StructField("altitude",   DoubleType(), True),
    StructField("utc_offset", DoubleType(), True),
    StructField("dst",        StringType(), True),
    StructField("timezone",   StringType(), True),
    StructField("type",       StringType(), True),
    StructField("source",     StringType(), True)
])

In [0]:
from pyspark.sql import functions as F
df_airports = (
    spark.read.option("multiline", True) \
        .json("/Volumes/us_project_catalog/raw/airpot_data/airports.json") \
        .filter(F.col("latitude").cast("double").between(-90, 90))    # DQ: valid lat
        .filter(F.col("longitude").cast("double").between(-180, 180)) # DQ: valid lon
        .withColumn("_load_ts", F.current_timestamp())
)

df_airports.count()

display(df_airports)

altitude,city,country,dst,iata,icao,id,latitude,longitude,name,source,timezone,type,utc_offset,_load_ts
5282,Goroka,Papua New Guinea,U,GKA,AYGA,1,-6.081689834590001,145.391998291,Goroka Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-15T09:13:12.494Z
20,Madang,Papua New Guinea,U,MAG,AYMD,2,-5.20707988739,145.789001465,Madang Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-15T09:13:12.494Z
5388,Mount Hagen,Papua New Guinea,U,HGU,AYMH,3,-5.826789855957031,144.29600524902344,Mount Hagen Kagamuga Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-15T09:13:12.494Z
239,Nadzab,Papua New Guinea,U,LAE,AYNZ,4,-6.569803,146.725977,Nadzab Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-15T09:13:12.494Z
146,Port Moresby,Papua New Guinea,U,POM,AYPY,5,-9.443380355834961,147.22000122070312,Port Moresby Jacksons International Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-15T09:13:12.494Z
19,Wewak,Papua New Guinea,U,WWK,AYWK,6,-3.58383011818,143.669006348,Wewak International Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-15T09:13:12.494Z
112,Narssarssuaq,Greenland,E,UAK,BGBW,7,61.1604995728,-45.4259986877,Narsarsuaq Airport,OurAirports,America/Godthab,airport,-3,2026-09-15T09:13:12.494Z
283,Godthaab,Greenland,E,GOH,BGGH,8,64.19090271,-51.6781005859,Godthaab / Nuuk Airport,OurAirports,America/Godthab,airport,-3,2026-09-15T09:13:12.494Z
165,Sondrestrom,Greenland,E,SFJ,BGSF,9,67.0122218992,-50.7116031647,Kangerlussuaq Airport,OurAirports,America/Godthab,airport,-3,2026-09-15T09:13:12.494Z
251,Thule,Greenland,E,THU,BGTL,10,76.5311965942,-68.7032012939,Thule Air Base,OurAirports,America/Thule,airport,-4,2026-09-15T09:13:12.494Z


In [0]:
df_airports.filter(F.col("country") == "United States").show()

+--------+----------------+-------------+---+----+----+----+------------------+-------------------+--------------------+-----------+-----------------+-------+----------+--------------------+
|altitude|            city|      country|dst|iata|icao|  id|          latitude|          longitude|                name|     source|         timezone|   type|utc_offset|            _load_ts|
+--------+----------------+-------------+---+----+----+----+------------------+-------------------+--------------------+-----------+-----------------+-------+----------+--------------------+
|       2|   Barter Island|United States|  A| BTI|PABA|3411|     70.1340026855|     -143.582000732|Barter Island LRR...|OurAirports|America/Anchorage|airport|        -9|2026-09-15 09:13:...|
|      16|   Cape Lisburne|United States|  A| LUR|PALU|3413|       68.87509918|       -166.1100006|Cape Lisburne LRR...|OurAirports|America/Anchorage|airport|        -9|2026-09-15 09:13:...|
|      22|       Point Lay|United States|  A|

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
TABLE_NAME = "us_project_catalog.raw.Airport_delta_table"


df_airports.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)   
  
delta_table = DeltaTable.forName(spark, TABLE_NAME)

spark.read.table(TABLE_NAME).show()



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7213051977907137>, line 4
      1 TABLE_NAME = "us_project_catalog.raw.Airport_delta_table"
----> 4 df_airports.write.format("delta").saveAsTable(TABLE_NAME)  
      7 delta_table = DeltaTable.forName(spark, TABLE_NAME)
      9 spark.read.table(TABLE_NAME).show()

File <command-5874586242882790>, line 3
      1 from pyspark.sql import functions as F
      2 df_airports = (
----> 3     spark.read.option("multiline", True) \
      4         .json("/Volumes/us_project_catalog/raw/airpot_data/airports.json") \
      5         .filter(F.col("latitude").cast("double").between(-90, 90))    # DQ: valid lat
      6         .filter(F.col("longitude").cast("double").between(-180, 180)) # DQ: valid lon
      7         .withColumn("_load_ts", F.current_timestamp())
      8 )
     10 df_airports.count()
     12 display(df_airports)

Ana

In [0]:
new_airport_df = spark.read.table(TABLE_NAME)

In [0]:
new_airport_df.display()

altitude,city,country,dst,iata,icao,id,latitude,longitude,name,source,timezone,type,utc_offset,_load_ts
5282,Goroka,Papua New Guinea,U,GKA,AYGA,1,-6.081689834590001,145.391998291,Goroka Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-10T11:05:41.221Z
20,Madang,Papua New Guinea,U,MAG,AYMD,2,-5.20707988739,145.789001465,Madang Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-10T11:05:41.221Z
5388,Mount Hagen,Papua New Guinea,U,HGU,AYMH,3,-5.826789855957031,144.29600524902344,Mount Hagen Kagamuga Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-10T11:05:41.221Z
239,Nadzab,Papua New Guinea,U,LAE,AYNZ,4,-6.569803,146.725977,Nadzab Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-10T11:05:41.221Z
146,Port Moresby,Papua New Guinea,U,POM,AYPY,5,-9.443380355834961,147.22000122070312,Port Moresby Jacksons International Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-10T11:05:41.221Z
19,Wewak,Papua New Guinea,U,WWK,AYWK,6,-3.58383011818,143.669006348,Wewak International Airport,OurAirports,Pacific/Port_Moresby,airport,10,2026-09-10T11:05:41.221Z
112,Narssarssuaq,Greenland,E,UAK,BGBW,7,61.1604995728,-45.4259986877,Narsarsuaq Airport,OurAirports,America/Godthab,airport,-3,2026-09-10T11:05:41.221Z
283,Godthaab,Greenland,E,GOH,BGGH,8,64.19090271,-51.6781005859,Godthaab / Nuuk Airport,OurAirports,America/Godthab,airport,-3,2026-09-10T11:05:41.221Z
165,Sondrestrom,Greenland,E,SFJ,BGSF,9,67.0122218992,-50.7116031647,Kangerlussuaq Airport,OurAirports,America/Godthab,airport,-3,2026-09-10T11:05:41.221Z
251,Thule,Greenland,E,THU,BGTL,10,76.5311965942,-68.7032012939,Thule Air Base,OurAirports,America/Thule,airport,-4,2026-09-10T11:05:41.221Z
